In [681]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns  # 导入Seaborn库，用于统计数据可视化
plt.rcParams['font.sans-serif'] = ['SimHei']
import xgboost as xgb
import lightgbm as lgb 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score

1. 数据加载

In [682]:
train_data = pd.read_csv('health_lifestyle_classification.csv',encoding='utf-8')

In [683]:
train_data

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,...,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target
0,1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,...,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502,healthy
1,2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,...,High,5,High,Yes,No,0,1.0,5.5,6.239340,healthy
2,3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,...,High,4,Moderate,No,No,0,1.0,5.5,5.423737,healthy
3,4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,...,High,1,NaN,No,Yes,0,1.0,5.5,8.388611,healthy
4,5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,...,High,1,High,Yes,Yes,0,1.0,5.5,0.332622,healthy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,53,Male,177.202253,54.303671,17.293811,17.293811,51.881433,17.227616,88.740028,...,Moderate,1,High,No,Yes,0,1.0,5.5,3.477124,healthy
99996,99997,22,Male,180.802297,40.033853,12.246712,12.246712,36.740135,12.159473,103.659560,...,Moderate,5,NaN,Yes,No,0,1.0,5.5,1.609656,healthy
99997,99998,37,Male,185.540653,84.536847,24.556580,24.556580,73.669741,24.172944,98.920422,...,Low,5,NaN,No,No,0,1.0,5.5,-9.736463,healthy
99998,99999,72,Female,181.796786,56.923335,17.223362,17.223362,51.670087,17.715475,54.559079,...,Low,4,High,Yes,Yes,0,1.0,5.5,-4.779376,healthy


In [684]:
# 分离目标变量(方法1)
y = train_data.pop('target')   # 删除列名指定的列,同时将结果返回到y_train

In [685]:
# 初步检查
# print(f"数据维度: {train_df.shape}")
print("缺失值统计:\n", train_data.isnull().sum().sort_values(ascending=False).head(10))

缺失值统计:
 alcohol_consumption    42387
caffeine_intake        33261
exercise_type          24969
insulin                15836
heart_rate             14003
gene_marker_flag       10474
income                  8470
daily_steps             8329
blood_pressure          7669
survey_code                0
dtype: int64


In [686]:
# 通过set_index()将第一列ID设为索引列，规范数据格式
train_df = train_data.set_index(train_data.columns[0])  # set_index设置新的索引，并删除该列

In [687]:
train_df

,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,...,insurance,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage
survey_code,,,,,,,,,,,,,,,,,,,,,
1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,118.264254,...,No,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502
2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,117.917986,...,No,High,5,High,Yes,No,0,1.0,5.5,6.239340
3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,123.073698,...,Yes,High,4,Moderate,No,No,0,1.0,5.5,5.423737
4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,148.173453,...,No,High,1,NaN,No,Yes,0,1.0,5.5,8.388611
5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,150.613181,...,Yes,High,1,High,Yes,Yes,0,1.0,5.5,0.332622
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99996,53,Male,177.202253,54.303671,17.293811,17.293811,51.881433,17.227616,88.740028,135.090834,...,No,Moderate,1,High,No,Yes,0,1.0,5.5,3.477124
99997,22,Male,180.802297,40.033853,12.246712,12.246712,36.740135,12.159473,103.659560,135.181795,...,No,Moderate,5,NaN,Yes,No,0,1.0,5.5,1.609656
99998,37,Male,185.540653,84.536847,24.556580,24.556580,73.669741,24.172944,98.920422,146.504768,...,Yes,Low,5,NaN,No,No,0,1.0,5.5,-9.736463


## 一、数据预处理

### 1. 特征类型识别与处理

In [688]:
# 识别数值型特征
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"数值型特征数量: {len(numeric_cols)}")

数值型特征数量: 29


In [689]:
print(numeric_cols)

['age', 'height', 'weight', 'bmi', 'bmi_estimated', 'bmi_scaled', 'bmi_corrected', 'waist_size', 'blood_pressure', 'heart_rate', 'cholesterol', 'glucose', 'insulin', 'sleep_hours', 'work_hours', 'physical_activity', 'daily_steps', 'calorie_intake', 'sugar_intake', 'water_intake', 'screen_time', 'stress_level', 'mental_health_score', 'income', 'meals_per_day', 'electrolyte_level', 'gene_marker_flag', 'environmental_risk_score', 'daily_supplement_dosage']


In [690]:
# 识别潜在的分类特征（唯一值较少的数值列）
potential_categorical = []
true_numeric_cols = []  # 真正的连续数值特征

for col in numeric_cols:
    if train_df[col].nunique() <= 11 and train_df[col].nunique() > 1:
        potential_categorical.append(col)
    else:
        true_numeric_cols.append(col)
        
print(f"潜在数值型分类特征: {len(potential_categorical)}")
print(f"真正的连续数值特征: {len(true_numeric_cols)}")

潜在数值型分类特征: 3
真正的连续数值特征: 26


In [691]:
print(potential_categorical)

['stress_level', 'mental_health_score', 'meals_per_day']


In [692]:
# 识别文本型特征
text_cols = train_df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"文本型特征数量: {len(text_cols)}")

文本型特征数量: 17


### 3.数据转换
将数值类型列保留两位小数

In [693]:
train_df[true_numeric_cols] = train_df[true_numeric_cols].astype(float).round(2)

In [694]:
train_df

,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,...,insurance,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage
survey_code,,,,,,,,,,,,,,,,,,,,,
1,56.0,Male,173.42,56.89,18.92,18.92,56.75,18.99,72.17,118.26,...,No,High,5,Moderate,No,Yes,0.0,1.0,5.5,-2.28
2,69.0,Female,163.21,97.80,36.72,36.72,110.15,36.51,85.60,117.92,...,No,High,5,High,Yes,No,0.0,1.0,5.5,6.24
3,46.0,Male,177.28,80.69,25.67,25.67,77.02,25.59,90.30,123.07,...,Yes,High,4,Moderate,No,No,0.0,1.0,5.5,5.42
4,32.0,Female,172.10,63.14,21.32,21.32,63.96,21.18,100.50,148.17,...,No,High,1,NaN,No,Yes,0.0,1.0,5.5,8.39
5,60.0,Female,163.61,40.00,14.94,14.94,44.83,14.84,69.02,150.61,...,Yes,High,1,High,Yes,Yes,0.0,1.0,5.5,0.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99996,53.0,Male,177.20,54.30,17.29,17.29,51.88,17.23,88.74,135.09,...,No,Moderate,1,High,No,Yes,0.0,1.0,5.5,3.48
99997,22.0,Male,180.80,40.03,12.25,12.25,36.74,12.16,103.66,135.18,...,No,Moderate,5,NaN,Yes,No,0.0,1.0,5.5,1.61
99998,37.0,Male,185.54,84.54,24.56,24.56,73.67,24.17,98.92,146.50,...,Yes,Low,5,NaN,No,No,0.0,1.0,5.5,-9.74


### 2. 缺失值处理

In [695]:
# 真正的连续数值特征用中位数填充
if true_numeric_cols:
    train_df[true_numeric_cols] = train_df[true_numeric_cols].fillna(train_df[true_numeric_cols].median())
    print("连续数值特征缺失值已用中位数填充")

连续数值特征缺失值已用中位数填充


In [696]:
# 数值型分类特征用众数填充
if potential_categorical:
    for col in potential_categorical:
        if train_df[col].notna().any():
            # 对于数值型分类，用众数填充
            mode_val = train_df[col].mode()[0] if not train_df[col].mode().empty else train_df[col].median()
            train_df[col] = train_df[col].fillna(mode_val)
    print("数值型分类特征缺失值已用众数填充")

数值型分类特征缺失值已用众数填充


In [697]:
# 文本型特征用众数填充
if text_cols:
    for col in text_cols:
        if train_df[col].notna().any():
            mode_val = train_df[col].mode()[0] if not train_df[col].mode().empty else "MISSING"
            train_df[col] = train_df[col].fillna(mode_val)
    print("文本型特征缺失值已用众数填充")

文本型特征缺失值已用众数填充


In [698]:
train_df.shape

(100000, 46)

In [699]:
# 检查是否填充完整
print("缺失值统计:\n", train_df.isnull().sum().sort_values(ascending=False).head(10))

缺失值统计:
 age                      0
device_usage             0
stress_level             0
mental_health_score      0
mental_health_support    0
education_level          0
job_type                 0
occupation               0
income                   0
diet_type                0
dtype: int64


### 4. 特征编码

In [700]:
# 文本特征进行One-Hot编码
if text_cols:
    train_df_new = pd.get_dummies(train_df, columns=text_cols, prefix_sep="::")
    print(f"文本特征One-Hot编码后新增 {len(train_df_new.columns) - len(train_df.columns)} 列")

文本特征One-Hot编码后新增 36 列


### 4. 特征选择

In [701]:
# # 移除低方差特征
# from sklearn.feature_selection import VarianceThreshold

# selector = VarianceThreshold(threshold=0.1)
# X_selected = selector.fit_transform(train_df_new)   # 移除低方差特征后的数据 ---> array（数组）

In [702]:
# 移除低方差特征
from sklearn.feature_selection import VarianceThreshold

# 移除低方差特征（阈值0.1）
selector = VarianceThreshold(threshold=0.1)
X_selected = selector.fit_transform(train_df_new)

# 重建DataFrame
selected_columns = train_df_new.columns[selector.get_support()]
train_df_selected = pd.DataFrame(X_selected, columns=selected_columns)

print("\n移除低方差特征后的DataFrame:")
print(train_df_selected.shape)


移除低方差特征后的DataFrame:
(100000, 79)


In [703]:
print(f"特征选择前: {train_df_new.shape[1]} 个特征")
print(f"特征选择后: {train_df_selected.shape[1]} 个特征")
print(f"移除了 {train_df_new.shape[1] - train_df_selected.shape[1]} 个低方差特征")

特征选择前: 82 个特征
特征选择后: 79 个特征
移除了 3 个低方差特征


In [704]:
type(train_df_selected)

pandas.core.frame.DataFrame

In [705]:
train_df_selected

,age,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,heart_rate,...,insurance::Yes,sunlight_exposure::High,sunlight_exposure::Low,sunlight_exposure::Moderate,caffeine_intake::High,caffeine_intake::Moderate,family_history::No,family_history::Yes,pet_owner::No,pet_owner::Yes
0,56.0,173.42,56.89,18.92,18.92,56.75,18.99,72.17,118.26,60.75,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
1,69.0,163.21,97.80,36.72,36.72,110.15,36.51,85.60,117.92,66.46,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
2,46.0,177.28,80.69,25.67,25.67,77.02,25.59,90.30,123.07,76.04,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
3,32.0,172.10,63.14,21.32,21.32,63.96,21.18,100.50,148.17,68.78,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4,60.0,163.61,40.00,14.94,14.94,44.83,14.84,69.02,150.61,92.34,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,53.0,177.20,54.30,17.29,17.29,51.88,17.23,88.74,135.09,75.34,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
99996,22.0,180.80,40.03,12.25,12.25,36.74,12.16,103.66,135.18,56.33,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
99997,37.0,185.54,84.54,24.56,24.56,73.67,24.17,98.92,146.50,74.86,...,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
99998,72.0,181.80,56.92,17.22,17.22,51.67,17.72,54.56,100.99,64.72,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


In [706]:
# 获取保留的特征名称
selected_features = train_df_new.columns[selector.get_support()].tolist()
selected_features

['age',
 'height',
 'weight',
 'bmi',
 'bmi_estimated',
 'bmi_scaled',
 'bmi_corrected',
 'waist_size',
 'blood_pressure',
 'heart_rate',
 'cholesterol',
 'glucose',
 'insulin',
 'sleep_hours',
 'work_hours',
 'physical_activity',
 'daily_steps',
 'calorie_intake',
 'sugar_intake',
 'water_intake',
 'screen_time',
 'stress_level',
 'mental_health_score',
 'income',
 'meals_per_day',
 'daily_supplement_dosage',
 'gender::Female',
 'gender::Male',
 'sleep_quality::Excellent',
 'sleep_quality::Fair',
 'sleep_quality::Good',
 'sleep_quality::Poor',
 'alcohol_consumption::Occasionally',
 'alcohol_consumption::Regularly',
 'smoking_level::Heavy',
 'smoking_level::Light',
 'smoking_level::Non-smoker',
 'mental_health_support::No',
 'mental_health_support::Yes',
 'education_level::Bachelor',
 'education_level::High School',
 'education_level::Master',
 'education_level::PhD',
 'job_type::Healthcare',
 'job_type::Labor',
 'job_type::Office',
 'job_type::Service',
 'job_type::Tech',
 'job_type::

In [707]:
len(selected_features)

79

## 当前数据集：

In [708]:
# 特征：train_df_selected
# 标签：y

# 划分数据集

In [709]:
X = train_df_selected.values
le = LabelEncoder()
y = le.fit_transform(y)
y


array([1, 1, 1, ..., 1, 1, 0])

# 将特征列与目标列分离

In [710]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 定义并训练XGBoost模型

In [711]:
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

# 定义并训练LightGBM模型

In [712]:
lgb_model = lgb.LGBMClassifier(
    objective='binary',
    metric='binary_logloss',
    random_state=42,
    force_col_wise=True,
    verbosity=0 
)
lgb_model.fit(X_train, y_train)


LGBMClassifier(force_col_wise=True, metric='binary_logloss', objective='binary',
               random_state=42, verbosity=0)

# 预测与评估

In [713]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name, cv=5):
    
    # 在测试集上评估
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    print(f'\n{model_name} 测试集评估结果:')
    print(f'准确率: {accuracy:.4f}')
    print(f'精确率: {precision:.4f}')
    print(f'召回率: {recall:.4f}')
    
    # 交叉验证评估
    print(f'\n{model_name} 交叉验证结果 (cv={cv}):')
    # 交叉验证准确率
    cv_accuracy = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    print(f'交叉验证准确率: {cv_accuracy.mean():.4f} (±{cv_accuracy.std():.4f})')
    
    # 交叉验证精确率
    cv_precision = cross_val_score(model, X_train, y_train, cv=cv, scoring='precision')
    print(f'交叉验证精确率: {cv_precision.mean():.4f} (±{cv_precision.std():.4f})')
    
    # 交叉验证召回率
    cv_recall = cross_val_score(model, X_train, y_train, cv=cv, scoring='recall')
    print(f'交叉验证召回率: {cv_recall.mean():.4f} (±{cv_recall.std():.4f})')
    
    # 返回所有评估指标
    return {
        'test_accuracy': accuracy,
        'test_precision': precision,
        'test_recall': recall,
        'cv_accuracy': (cv_accuracy.mean(), cv_accuracy.std()),
        'cv_precision': (cv_precision.mean(), cv_precision.std()),
        'cv_recall': (cv_recall.mean(), cv_recall.std())
    }

In [714]:

evaluate_model(xgb_model, X_train, y_train, X_test, y_test, 'XGBoost')
evaluate_model(lgb_model, X_train, y_train, X_test, y_test, 'LightGBM')
use_label_encoder=False 



XGBoost 测试集评估结果:
准确率: 0.6866
精确率: 0.6984
召回率: 0.9708

XGBoost 交叉验证结果 (cv=5):
交叉验证准确率: 0.6861 (±0.0018)
交叉验证精确率: 0.7009 (±0.0006)
交叉验证召回率: 0.9640 (±0.0027)

LightGBM 测试集评估结果:
准确率: 0.6988
精确率: 0.6988
召回率: 0.9999

LightGBM 交叉验证结果 (cv=5):
交叉验证准确率: 0.7014 (±0.0001)
交叉验证精确率: 0.7016 (±0.0000)
交叉验证召回率: 0.9996 (±0.0003)
